# GPU Voxelization Demo (Length-Weighted)

This notebook demonstrates the updated GPU-only voxelization pipeline. The CUDA kernel now allocates charge to voxels proportionally to the actual fraction of each track segment's length inside the voxel (3D DDA traversal), replacing the previous uniform-per-step approximation.

Highlights:
- Length-weighted charge deposition per voxel (conserves total electrons).
- Event/TPC scoping consistent with main pipeline.
- Edge cases: single-voxel segments, multi-voxel partial overlaps, diagonal crossing.
- Performance note: very small test samples show low GPU occupancy warnings (harmless for correctness).

We'll visualize charge distribution and validate fractional splits against analytical expectations.


In [ ]:
# Section 1: Import Required Libraries (with path setup)
import os
import sys
import pathlib
import numpy as np
import h5py
import matplotlib.pyplot as plt
from math import ceil

# Ensure project root is on sys.path for local package imports
NOTEBOOK_DIR = pathlib.Path(__file__).resolve().parent
REPO_ROOT = NOTEBOOK_DIR.parent  # larnd-sim root
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

try:
    import cupy as cp
except ImportError:
    raise RuntimeError("CuPy is required for GPU voxelization demo.")

# Attempt imports; provide diagnostic if fails
try:
    from larndsim import quenching, drifting, consts
    from larndsim.active_volume import select_active_volume
    from larndsim.mesh_refinement.voxelization import gpu_voxelize
    from larndsim.consts import sim, physics
    from larndsim.config import get_config
except ModuleNotFoundError as e:
    print("Import error:", e)
    print("sys.path:")
    for p in sys.path: print("  ", p)
    raise

plt.rcParams['figure.figsize'] = (7,5)


In [ ]:
# Section 2: Load or Create 3D Data (segments)
EXAMPLE_FILE = os.path.join(os.path.dirname(__file__), 'lbnfSpillLAr.edep.h5')
if not os.path.exists(EXAMPLE_FILE):
    raise FileNotFoundError(f"Example file not found: {EXAMPLE_FILE}")

with h5py.File(EXAMPLE_FILE, 'r') as f:
    segments = np.array(f['segments'])

print(f"Loaded {segments.shape[0]} raw segments")

# Add missing timing fields if absent (older inputs)
if 't0' not in segments.dtype.names:
    t0 = np.array(segments['t'].copy(), dtype=[('t0','f4')])
    t0_start = np.array(segments['t_start'].copy(), dtype=[('t0_start','f4')])
    t0_end = np.array(segments['t_end'].copy(), dtype=[('t0_end','f4')])
    segments = np.lib.recfunctions.merge_arrays((segments, t0, t0_start, t0_end), flatten=True)
    segments['t'] = np.zeros(segments.shape[0], dtype=[('t','f4')])
    segments['t_start'] = np.zeros(segments.shape[0], dtype=[('t_start','f4')])
    segments['t_end'] = np.zeros(segments.shape[0], dtype=[('t_end','f4')])

# Add segment_id if missing
if 'segment_id' not in segments.dtype.names:
    dtype = [('segment_id','u4')] + segments.dtype.descr
    new = np.empty(segments.shape, dtype=np.dtype(dtype, align=True))
    new['segment_id'] = np.arange(segments.shape[0], dtype='u4')
    for name, fmt in segments.dtype.descr:
        new[name] = segments[name]
    segments = new

print("Fields present:", segments.dtype.names)

In [ ]:
# Section 3: Initialize Voxelization Parameters & Preprocessing
from larndsim.consts import detector  # ensure module is accessible after property load

# Load configuration (2x2 assumed) and detector properties
cfg = get_config('2x2')
consts.load_properties(cfg['DET_PROPERTIES'], cfg['PIXEL_LAYOUT'], cfg['RESPONSE'], cfg['SIM_PROPERTIES'])
from larndsim.consts import detector  # reload after properties

# Filter neutrals (neutrons 2112, gammas 22)
mask_neutral = (segments['pdg_id'] != 2112) & (segments['pdg_id'] != 22)
segments = segments[mask_neutral]
print(f"After neutral filtering: {segments.shape[0]} segments")

# Filter delayed by t0
if 't0' in segments.dtype.names:
    segments = segments[segments['t0'] < sim.MAX_SEGMENT_T0]
    print(f"After delayed filter: {segments.shape[0]} segments")

# First event only
first_event = np.min(segments[sim.EVENT_SEPARATOR])
segments = segments[segments[sim.EVENT_SEPARATOR] == first_event]
print(f"Segments in first event: {segments.shape[0]}")

# Swap x and z to larnd-sim convention
x_start = np.copy(segments['x_start']); x_end = np.copy(segments['x_end'])
if 'x' in segments.dtype.names: x_mid = np.copy(segments['x'])
segments['x_start'] = np.copy(segments['z_start']); segments['x_end'] = np.copy(segments['z_end'])
if 'x' in segments.dtype.names: segments['x'] = np.copy(segments['z'])
segments['z_start'] = x_start; segments['z_end'] = x_end
if 'x' in segments.dtype.names: segments['z'] = x_mid
print("Coordinate swap complete (edep -> larnd-sim).")

# Select first TPC volume
first_tpc_bounds = detector.TPC_BORDERS[0:1]
idx_tpc = select_active_volume(segments, first_tpc_bounds)
sel_tracks = segments[idx_tpc]
print(f"Tracks in first TPC: {sel_tracks.shape[0]}")

# Subsample for speed if large
if sel_tracks.shape[0] > 5000:
    sel_tracks = sel_tracks[:5000]
    print("Subsampled to 5000 tracks for demo speed.")

# Quenching + drifting
TPB = 256
BPG = max(ceil(sel_tracks.shape[0] / TPB), 1)
quenching.quench[BPG, TPB](sel_tracks, physics.BIRKS)
drifting.drift[BPG, TPB](sel_tracks)
assert 'n_electrons' in sel_tracks.dtype.names
print("Applied quenching and drifting kernels.")

In [ ]:
# Section 4: Perform Voxelization (GPU)
nonzero_idx, nonzero_charge, grid_shape, voxel_size, bounds = gpu_voxelize(sel_tracks, tpc_borders=first_tpc_bounds)

print("Voxelization complete")
print("Grid shape (nx, ny, nz):", grid_shape)
print("Voxel size (cm):", voxel_size)
print("Occupied voxels:", nonzero_idx.size)

# Charge conservation check
track_charge = float(np.sum(sel_tracks['n_electrons']))
voxel_charge = float(np.sum(nonzero_charge.get() if hasattr(nonzero_charge, 'get') else nonzero_charge))
rel_diff = abs(voxel_charge - track_charge) / track_charge if track_charge > 0 else 0.0
print(f"Total track charge: {track_charge:.3e} e-")
print(f"Total voxel charge: {voxel_charge:.3e} e- (rel diff {rel_diff:.2e})")

In [ ]:
# Section 5: Helper to decode flattened voxel indices -> (ix, iy, iz)
def decode_indices(flat_indices, grid_shape):
    nx, ny, nz = grid_shape
    iz = flat_indices // (nx * ny)
    rem = flat_indices % (nx * ny)
    iy = rem // nx
    ix = rem % nx
    return ix, iy, iz

ix, iy, iz = decode_indices(nonzero_idx.get() if hasattr(nonzero_idx, 'get') else nonzero_idx, grid_shape)

# Compute voxel center world coordinates
x_min, x_max = bounds[0]; y_min, y_max = bounds[1]; z_min, z_max = bounds[2]
xc = x_min + (ix + 0.5) * voxel_size[0]
yc = y_min + (iy + 0.5) * voxel_size[1]
zc = z_min + (iz + 0.5) * voxel_size[2]

# Convert charge array to host for plotting
charge_host = nonzero_charge.get() if hasattr(nonzero_charge, 'get') else nonzero_charge


In [ ]:
# Section 6: Visualize Voxelized Output (2D projections)
fig, axes = plt.subplots(1, 3, figsize=(15,4))

# XY scatter color=log10(charge)
sc = axes[0].scatter(xc, yc, c=np.log10(charge_host+1e-6), s=5, cmap='viridis')
axes[0].set_title('XY Charge Density (log10)')
axes[0].set_xlabel('X [cm]'); axes[0].set_ylabel('Y [cm]')
plt.colorbar(sc, ax=axes[0], label='log10(charge e-)')

# Z histogram (occupancy per drift slice)
axes[1].hist(zc, bins=min(50, len(np.unique(iz))), color='steelblue', alpha=0.8)
axes[1].set_title('Voxel Occupancy vs Z')
axes[1].set_xlabel('Z [cm]'); axes[1].set_ylabel('Count')

# Charge distribution
axes[2].hist(np.log10(charge_host+1e-6), bins=50, color='darkorange', alpha=0.8)
axes[2].set_title('Voxel Charge Distribution')
axes[2].set_xlabel('log10(charge e-)'); axes[2].set_ylabel('Voxels')

plt.tight_layout()
plt.show()


In [ ]:
# Section 7: Compare Original vs Voxelized (XY overlay sample)
# Take a subset of original segment start points for overlay
n_sample = min(2000, sel_tracks.shape[0])
xs = sel_tracks['x_start'][:n_sample]
ys = sel_tracks['y_start'][:n_sample]

plt.figure(figsize=(6,5))
plt.scatter(xs, ys, s=4, c='gray', alpha=0.4, label='Segment starts')
plt.scatter(xc, yc, s=8, c=np.log10(charge_host+1e-6), cmap='viridis', alpha=0.9, label='Voxels')
plt.xlabel('X [cm]'); plt.ylabel('Y [cm]')
plt.title('Original Tracks vs Coarse Voxels (XY)')
plt.legend(loc='upper right')
plt.colorbar(label='log10(charge e-)')
plt.show()


In [ ]:
# Section 8: Experiment with Different Voxel Sizes
sizes = [voxel_size[0]*2, voxel_size[0], voxel_size[0]/2]
occ_fractions = []
for sx in sizes:
    test_voxel_size = (sx, sx, voxel_size[2])  # vary X/Y equally, keep Z
    nz_i, nz_q, gshape, vsize, bnds = gpu_voxelize(sel_tracks, tpc_borders=first_tpc_bounds, voxel_size=test_voxel_size)
    occ = nz_i.size / (gshape[0]*gshape[1]*gshape[2])
    occ_fractions.append((test_voxel_size, gshape, occ))

for vsize, gshape, occ in occ_fractions:
    print(f"Voxel size={vsize} -> grid={gshape}, occupancy={occ:.4f}")


## Next Steps & Extended Validation

New in this version:
- Length-weighted voxelization implemented in CUDA kernel.
- Added analytical comparison utilities for fractional charge splits.

Suggested explorations:
1. Increase segment count to observe occupancy scaling (avoid performance warning).
2. Benchmark against uniform-weighting approximation for large segment sets.
3. Integrate voxel output with downstream light/charge far-field calculators.
4. Add optional structured-dtype support if upstream data remains named.

Run the new validation cells below to see edge-case behavior.


### Fractional Charge Validation Cells
We add cells to:
1. Construct synthetic segments crossing multiple voxels.
2. Run GPU voxelization.
3. Compare measured voxel charges to analytical fractions.
4. Visualize discrepancies (should be ~0 within numerical tolerance).

In [ ]:
# Synthetic multi-voxel segments and analytical fraction calculator
import numpy as np
import cupy as cp
from larndsim.mesh_refinement.voxelization import gpu_voxelize

# Helper: compute expected fractional charge splits for axis-aligned x-only segment
# Returns dict voxel_index -> expected_charge

def expected_fraction_axis_x(x0, x1, q, voxel_size, x_min, n_x):
    if x1 < x0:
        x0, x1 = x1, x0
    total_len = x1 - x0
    out = {}
    if total_len <= 0:
        return out
    # Iterate voxel boundaries
    for ix in range(n_x):
        vx0 = x_min + ix * voxel_size[0]
        vx1 = vx0 + voxel_size[0]
        overlap = max(0.0, min(x1, vx1) - max(x0, vx0))
        if overlap > 0:
            out[ix] = q * (overlap / total_len)
    return out

# Example segments - use structured array to match simulate_pixels.py format
dtype = np.dtype([
    ('x_start', 'f4'), ('y_start', 'f4'), ('z_start', 'f4'),
    ('x_end', 'f4'), ('y_end', 'f4'), ('z_end', 'f4'),
    ('n_electrons', 'f4')
])
segments = np.array([
    (0.25, 0.5, 0.5, 1.75, 0.5, 0.5, 1000.0),  # crosses two voxels equally (should split 50/50)
    (0.9,  0.5, 0.5, 2.4,  0.5, 0.5, 900.0 ),  # unequal (0.1,1.0,0.4) length portions over 3 voxels
], dtype=dtype)

# Grid setup
voxel_size = (1.0,1.0,1.0)
tpc_borders = np.array([[[0,3],[0,1],[0,1]]], dtype=np.float32)

# Run voxelization
vox_idx, vox_charge, grid_shape, _, bounds = gpu_voxelize(segments, tpc_borders=tpc_borders, voxel_size=voxel_size)
np_idx = cp.asnumpy(vox_idx)
np_charge = cp.asnumpy(vox_charge)

print("Grid shape:", grid_shape)
print("Non-zero voxel count:", len(np_idx))

# Group charges per voxel
charges = {}
for vid, q in zip(np_idx, np_charge):
    charges[vid] = charges.get(vid, 0.0) + q

print("Voxel charges (flattened indices):", charges)


In [ ]:
# Compare against analytical expectations for each segment separately

# Reconstruct per-segment expected splits (for axis-aligned segments)
voxel_size = (1.0,1.0,1.0)
x_min = bounds[0][0]
expected_all = {}

# First segment (index 0)
seg0 = segments[0]
exp0 = expected_fraction_axis_x(seg0['x_start'], seg0['x_end'], seg0['n_electrons'], voxel_size, x_min, grid_shape[0])
# Second segment (index 1)
seg1 = segments[1]
exp1 = expected_fraction_axis_x(seg1['x_start'], seg1['x_end'], seg1['n_electrons'], voxel_size, x_min, grid_shape[0])

print("Expected segment 0 splits:", exp0)
print("Expected segment 1 splits:", exp1)

# Combine expected totals per voxel across segments
combined_expected = {}
for d in (exp0, exp1):
    for k,v in d.items():
        combined_expected[k] = combined_expected.get(k,0.0) + v

# Show comparison
rows = []
for vid in sorted(combined_expected.keys() | charges.keys()):
    rows.append((vid, charges.get(vid,0.0), combined_expected.get(vid,0.0), charges.get(vid,0.0) - combined_expected.get(vid,0.0)))

print("VoxelIdx  Measured   Expected   Diff")
for r in rows:
    print(f"{r[0]:7d}  {r[1]:8.2f}  {r[2]:8.2f}  {r[3]:+8.3f}")


In [ ]:
# Visualization: bar chart of measured vs expected per voxel
import matplotlib.pyplot as plt

voxels = sorted(combined_expected.keys() | charges.keys())
measured = [charges.get(v,0.0) for v in voxels]
expected = [combined_expected.get(v,0.0) for v in voxels]

x = np.arange(len(voxels))
width = 0.35
plt.figure(figsize=(8,4))
plt.bar(x - width/2, measured, width, label='Measured')
plt.bar(x + width/2, expected, width, label='Expected')
plt.xticks(x, voxels)
plt.xlabel('Voxel Index (flattened along x)')
plt.ylabel('Charge (electrons)')
plt.title('Length-Weighted Voxel Charge Distribution')
plt.legend()
plt.show()

# Assert closeness numerically
for v, m, e in zip(voxels, measured, expected):
    assert np.isclose(m, e, rtol=1e-3, atol=1e-2), f"Voxel {v} mismatch: {m} vs {e}"
print("All voxel charges match analytical expectations within tolerance.")

In [ ]:
# Optional: occupancy scaling demo (no correctness change)
# Generate many short segments to show larger grid size -> higher occupancy
n_segments = 1024
rng = np.random.default_rng(42)
dtype = np.dtype([
    ('x_start', 'f4'), ('y_start', 'f4'), ('z_start', 'f4'),
    ('x_end', 'f4'), ('y_end', 'f4'), ('z_end', 'f4'),
    ('n_electrons', 'f4')
])
short_segments = []
for _ in range(n_segments):
    x0 = rng.uniform(0.0, 2.9)
    x1 = min(3.0, x0 + rng.uniform(0.05, 0.5))
    y = rng.uniform(0.0, 0.99)
    z = 0.5
    q = rng.uniform(50,150)
    short_segments.append((x0,y,z,x1,y,z,q))
short_segments = np.array(short_segments, dtype=dtype)

vox_idx2, vox_charge2, grid_shape2, _, _ = gpu_voxelize(short_segments, tpc_borders=tpc_borders, voxel_size=voxel_size)
print("Large batch non-zero voxels:", len(vox_idx2))
print("Kernel launch blocks (approx):", (short_segments.shape[0] + 256 - 1)//256)
